# 02a – Dashboard refactorizado de métricas (W_PCT & PLUS_MINUS)

**Propósito.** Revisar el dashboard de jugadores/equipos para entender qué métricas explican victorias e impacto.
**Dataset.** `team_player_dashboard__dataset_1.parquet` (fuente 00_data/00c_final/2024-25/dashboards).
**Targets.** `W_PCT` y `PLUS_MINUS`.

**Pipeline rápido**
- Carga del parquet y validación de columnas clave.
- Preparación ligera: tipado eficiente, categorías y saneo de nulos/duplicados básicos.
- Cálculo de correlaciones y métricas de soporte (KPIs ya usados en el cuaderno original).
- Visualizaciones compactas (máx. 3) y exporte ordenado de tablas/figuras.

> Refactorizado para priorizar simplicidad visual, celdas breves y salidas ordenadas.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

SAVE_FIG = True
FIG_DPI = 110
MAX_FIGS_PER_SECTION = 4

NOTEBOOK_DIR = Path.cwd()
OUTPUT_BASE = NOTEBOOK_DIR / "02a_dasboard_outs" / "02a_dashboard"
FIGURES_DIR = OUTPUT_BASE / "figures"
TABLES_DIR = OUTPUT_BASE / "tables"

DATA_PATH = Path("../../..") / "00_data/00c_final/2024-25/dashboards/team_player_dashboard__dataset_1.parquet"

## 1. Lectura de datos y validación

El análisis parte de un archivo `.parquet` accesible con ruta relativa dentro del repositorio. Se requieren, como mínimo, las columnas de identificación de equipo/jugador y los objetivos (`W_PCT`, `PLUS_MINUS`). Si alguna falta, las secciones dependientes quedan marcadas para omisión.

In [ ]:
REQUIRED_COLUMNS = {"TEAM_ID", "TEAM_NAME", "PLAYER_ID", "PLAYER_NAME", "W_PCT", "PLUS_MINUS"}
skip_analysis = False

if not DATA_PATH.exists():
    warnings.warn(f"Archivo .parquet no encontrado: {DATA_PATH}")
    df = pd.DataFrame()
    skip_analysis = True
else:
    df = pd.read_parquet(DATA_PATH)
    df.columns = [c.strip() for c in df.columns]
    missing_columns = REQUIRED_COLUMNS.difference(df.columns)

    print(f"Shape original: {df.shape}")
    display(df.head(3))

    dtype_summary = (
        df.dtypes.astype(str)
        .value_counts()
        .rename_axis("dtype")
        .to_frame("count")
    )
    display(dtype_summary)

    if missing_columns:
        warnings.warn(f"Columnas obligatorias ausentes: {sorted(missing_columns)}")
        skip_analysis = True

## 2. Preparación ligera y tipado eficiente

Se reconcilian columnas duplicadas (`TEAM_ID`/`team_id`, `SEASON_YEAR`/`season`), se filtra `dataset==1` si procede y se homogenizan tipos: numéricos en `float32/int32`, categóricos para textos frecuentes y sustitución de infinitos por `NaN`. El objetivo es mantener la lógica original con mejor eficiencia y validaciones ligeras.

In [ ]:
from typing import Dict, Iterable


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_filename(name: str) -> str:
    cleaned = [c if c.isalnum() or c in {"_", "-", "."} else "_" for c in name]
    return "".join(cleaned).strip("._").lower()


def save_figure(fig: plt.Figure, path: Path) -> None:
    if not SAVE_FIG:
        return
    ensure_dir(path.parent)
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")


def compute_spearman_correlations(data: pd.DataFrame, target: str, columns: Iterable[str]) -> pd.DataFrame:
    records = []
    for col in columns:
        if col == target or col not in data.columns:
            continue
        subset = data[[col, target]].dropna()
        if subset.shape[0] < 5:
            continue
        if subset[col].nunique(dropna=True) < 2 or subset[target].nunique(dropna=True) < 2:
            continue
        rho = subset[col].corr(subset[target], method="spearman")
        if pd.notna(rho):
            records.append({
                "variable": col,
                "rho_spearman": rho,
                "n_observaciones": int(subset.shape[0]),
            })
    if not records:
        return pd.DataFrame(columns=["variable", "rho_spearman", "n_observaciones"])
    return pd.DataFrame(records).sort_values("rho_spearman", ascending=False).reset_index(drop=True)

ensure_dir(FIGURES_DIR)
ensure_dir(TABLES_DIR)

In [ ]:
import io

if skip_analysis or df.empty:
    skip_analysis = True
    warnings.warn("No hay datos válidos para preparar.")
else:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    if {"TEAM_ID", "team_id"}.issubset(df.columns):
        df["TEAM_ID"] = df["TEAM_ID"].fillna(df["team_id"])
        df["team_id"] = df["team_id"].fillna(df["TEAM_ID"])
    elif "team_id" in df.columns and "TEAM_ID" not in df.columns:
        df["TEAM_ID"] = df["team_id"]
    elif "TEAM_ID" in df.columns and "team_id" not in df.columns:
        df["team_id"] = df["TEAM_ID"]

    if {"season", "SEASON_YEAR"}.issubset(df.columns):
        df["season"] = df["season"].fillna(df["SEASON_YEAR"])
        df["SEASON_YEAR"] = df["SEASON_YEAR"].fillna(df["season"])
    elif "SEASON_YEAR" in df.columns and "season" not in df.columns:
        df["season"] = df["SEASON_YEAR"]
    elif "season" in df.columns and "SEASON_YEAR" not in df.columns:
        df["SEASON_YEAR"] = df["season"]

    if "dataset" in df.columns:
        uniques = pd.Series(df["dataset"]).dropna().unique()
        if len(uniques) > 1 and 1 in uniques:
            before = df.shape
            df = df[df["dataset"].eq(1)].copy()
            print(f"Filtrado dataset==1: {before} -> {df.shape}")

    category_candidates = [
        "TEAM_ID",
        "TEAM_NAME",
        "PLAYER_ID",
        "PLAYER_NAME",
        "GROUP_SET",
        "season",
        "SEASON_YEAR",
        "season_type",
        "endpoint",
    ]
    for col in category_candidates:
        if col in df.columns:
            df[col] = df[col].astype("category")

    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="float")
        else:
            df[col] = pd.to_numeric(df[col], downcast="integer")

    rank_cols = [c for c in df.columns if c.endswith("_RANK")]
    target_cols = [c for c in ["W_PCT", "PLUS_MINUS"] if c in df.columns]
    numeric_features = [c for c in numeric_cols if c not in target_cols]

    buf = io.StringIO()
    df.info(buf=buf, memory_usage="deep")
    info_lines = buf.getvalue().splitlines()
    preview = info_lines[:min(15, len(info_lines))]
    print("
".join(preview))
    if len(info_lines) > len(preview):
        print("…")

## 3. Métricas disponibles y objetivos

Los objetivos analizados permanecen en `W_PCT` y `PLUS_MINUS`. Se reutilizan los mismos grupos de métricas del cuaderno original (producción, eficiencia, manejo de balón, rebote y defensa), encapsulados ahora en listas para reutilizarlos en los pasos siguientes. Se añaden `assert` suaves y advertencias para detectar ausencias antes de continuar.

In [ ]:
metric_groups = {
    "produccion": [
        "PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA"
    ],
    "eficiencia": [
        "FG_PCT", "FG3_PCT", "FT_PCT", "TS_PCT", "EFG_PCT"
    ],
    "manejo": [
        "AST", "TOV", "AST_TOV", "AST_RATIO"
    ],
    "rebote": [
        "OREB", "DREB", "REB"
    ],
    "defensa": [
        "STL", "BLK", "BLKA", "PF", "PFD"
    ],
}

metric_groups = {k: [c for c in v if not skip_analysis and c in df.columns] for k, v in metric_groups.items()}

if skip_analysis:
    numeric_metrics = []
else:
    numeric_metrics = sorted({c for cols in metric_groups.values() for c in cols} | set(numeric_features))
    numeric_metrics = [c for c in numeric_metrics if c not in target_cols]

if not target_cols:
    warnings.warn("Sin columnas objetivo disponibles; se omiten cálculos dependientes.")
    skip_analysis = True

print(f"Targets disponibles: {target_cols}")
print({k: len(v) for k, v in metric_groups.items()})
print(f"Métricas numéricas totales (sin targets): {len(numeric_metrics)}")

## 4. Correlaciones principales (Spearman)

Se mantiene Spearman para robustez frente a distribuciones no lineales. Se limita la salida a las Top-12 métricas por objetivo (ordenadas por `|rho|`) y se exporta una única tabla consolidada. Estos resultados alimentan las visualizaciones posteriores.

In [ ]:
corr_tables: Dict[str, pd.DataFrame] = {}
top_corr_for_heatmap: Dict[str, pd.Series] = {}
top_corr_records = []

if skip_analysis:
    warnings.warn("Correlaciones omitidas por falta de datos válidos.")
else:
    for target in target_cols:
        features = [c for c in numeric_metrics if c != target]
        corr_df = compute_spearman_correlations(df, target, features)
        if corr_df.empty:
            warnings.warn(f"Sin correlaciones válidas para {target}.")
            continue
        corr_tables[target] = corr_df

        ordered = corr_df.reindex(corr_df["rho_spearman"].abs().sort_values(ascending=False).index)
        top_n = ordered.head(12)
        top_corr_for_heatmap[target] = top_n.set_index("variable")["rho_spearman"]
        top_corr_records.append(top_n.assign(target=target))

        display(top_n)

    if top_corr_records:
        corr_export = pd.concat(top_corr_records, ignore_index=True)
        ensure_dir(TABLES_DIR)
        corr_export_path = TABLES_DIR / "correlations_top12.csv"
        corr_export.to_csv(corr_export_path, index=False)
        print(f"Tabla consolidada de correlaciones guardada en: {corr_export_path}")
    else:
        warnings.warn("No se generó tabla consolidada de correlaciones.")

### Datos preparados para visualizaciones

Se reutiliza únicamente el subconjunto de métricas con mayor `|rho|` por objetivo para mantener las gráficas compactas.

In [ ]:
if skip_analysis or not top_corr_for_heatmap:
    heatmap_df = pd.DataFrame()
else:
    heatmap_df = pd.DataFrame(top_corr_for_heatmap).sort_index()
    heatmap_df = heatmap_df.reindex(
        heatmap_df.abs().max(axis=1).sort_values(ascending=False).index
    )

## 5. Visualizaciones minimalistas

Configuración ligera de estilo “tipo LaTeX” y hasta tres figuras relevantes que resumen los hallazgos clave.

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

### Heatmap de correlaciones destacadas

In [ ]:
if skip_analysis or heatmap_df.empty:
    warnings.warn("No hay datos suficientes para el heatmap de correlaciones.")
else:
    fig, ax = plt.subplots(figsize=(6 * max(1, len(target_cols)), max(4, 0.45 * len(heatmap_df))))
    im = ax.imshow(heatmap_df.values, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns)
    ax.set_yticks(range(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)
    ax.set_title("Top correlaciones por objetivo")
    ax.set_xlabel("Objetivo")
    ax.set_ylabel("Métrica")

    for i in range(heatmap_df.shape[0]):
        for j in range(heatmap_df.shape[1]):
            value = heatmap_df.iloc[i, j]
            if not pd.isna(value):
                ax.text(j, i, f"{value:.2f}", ha="center", va="center", color="black")

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Rho Spearman")
    fig.tight_layout()
    save_figure(fig, FIGURES_DIR / "correlation_heatmap.png")
    plt.close(fig)

### Métrica más correlacionada vs. W_PCT

In [ ]:
if skip_analysis or "W_PCT" not in corr_tables:
    warnings.warn("No se puede graficar la relación con W_PCT.")
else:
    ordered = corr_tables["W_PCT"].reindex(
        corr_tables["W_PCT"]["rho_spearman"].abs().sort_values(ascending=False).index
    )
    best_metric = ordered.iloc[0]["variable"] if not ordered.empty else None

    if not best_metric:
        warnings.warn("No hay métrica principal para W_PCT.")
    else:
        data = df[[best_metric, "W_PCT"]].dropna()
        if data.empty:
            warnings.warn("Datos insuficientes para la dispersión principal.")
        else:
            fig, ax = plt.subplots(figsize=(6, 5))
            if data.shape[0] > 600:
                hb = ax.hexbin(data[best_metric], data["W_PCT"], gridsize=30, cmap="viridis", mincnt=1)
                cbar = fig.colorbar(hb, ax=ax)
                cbar.set_label("Densidad")
            else:
                ax.scatter(data[best_metric], data["W_PCT"], s=28, alpha=0.65, edgecolors="none")
            ax.set_title(f"{best_metric} vs W_PCT")
            ax.set_xlabel(best_metric)
            ax.set_ylabel("W_PCT")
            ax.grid(True, linestyle="--", alpha=0.3)
            fig.tight_layout()
            save_figure(fig, FIGURES_DIR / f"{safe_filename(best_metric)}_vs_wpct.png")
            plt.close(fig)

### Top jugadores por impacto acumulado

In [ ]:
if skip_analysis or not {"PLAYER_NAME", "TEAM_NAME", "PLUS_MINUS", "GP"}.issubset(df.columns):
    warnings.warn("No es posible calcular PLUS_MINUS por partido.")
else:
    df["PLUS_MINUS_PER_GAME"] = df["PLUS_MINUS"] / df["GP"].replace(0, np.nan)
    pm_data = df[["PLAYER_NAME", "TEAM_NAME", "PLUS_MINUS_PER_GAME"]].dropna()
    pm_data = pm_data.sort_values("PLUS_MINUS_PER_GAME", ascending=False).head(10)

    if pm_data.empty:
        warnings.warn("Sin jugadores suficientes para el ranking de impacto.")
    else:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.barh(pm_data["PLAYER_NAME"], pm_data["PLUS_MINUS_PER_GAME"], color="#1f77b4")
        ax.invert_yaxis()
        ax.set_title("Top 10 PLUS_MINUS por partido")
        ax.set_xlabel("PLUS_MINUS por partido")
        ax.set_ylabel("Jugador")
        for idx, value in enumerate(pm_data["PLUS_MINUS_PER_GAME"]):
            ax.text(value, idx, f" {value:.2f}", va="center", ha="left")
        fig.tight_layout()
        save_figure(fig, FIGURES_DIR / "top_plus_minus_per_game.png")
        plt.close(fig)

## 6. Resúmenes tabulares y exportes clave

Se conservan los KPIs esenciales del cuaderno original en tablas compactas: correlaciones top, ranking por `W_PCT`/impacto y alertas de desequilibrios jugador-equipo.

In [ ]:
summary_tables: Dict[str, pd.DataFrame] = {}

if skip_analysis:
    warnings.warn("Resúmenes omitidos por falta de datos.")
else:
    if {"TEAM_NAME", "W_PCT"}.issubset(df.columns):
        team_summary = (
            df.groupby("TEAM_NAME", observed=True)[[c for c in ["W_PCT", "PLUS_MINUS"] if c in df.columns]]
            .mean()
            .sort_values("W_PCT", ascending=False)
        )
        summary_tables["team_summary"] = team_summary
        display(team_summary.head(10))

    if "PLUS_MINUS_PER_GAME" not in df.columns and {"PLUS_MINUS", "GP"}.issubset(df.columns):
        df["PLUS_MINUS_PER_GAME"] = df["PLUS_MINUS"] / df["GP"].replace(0, np.nan)

    insight_rows = []
    if {"PLAYER_NAME", "TEAM_NAME", "PTS", "W_PCT"}.issubset(df.columns):
        temp = df[["PLAYER_NAME", "TEAM_NAME", "PTS", "W_PCT"]].dropna()
        temp = temp.sort_values(["PTS", "W_PCT"], ascending=[False, True]).head(10)
        temp.insert(0, "insight", "PTS_alto_WPCT_bajo")
        insight_rows.append(temp)

    if {"PLAYER_NAME", "TEAM_NAME", "MIN", "PLUS_MINUS_PER_GAME"}.issubset(df.columns):
        temp = df[["PLAYER_NAME", "TEAM_NAME", "MIN", "PLUS_MINUS_PER_GAME"]].dropna()
        temp = temp.sort_values(["MIN", "PLUS_MINUS_PER_GAME"], ascending=[False, True]).head(10)
        temp.insert(0, "insight", "MIN_alto_PMpg_bajo")
        insight_rows.append(temp)

    if insight_rows:
        player_insights = pd.concat(insight_rows, ignore_index=True)
        summary_tables["player_insights"] = player_insights
        display(player_insights)

    for name, table in summary_tables.items():
        out_path = TABLES_DIR / f"{name}.csv"
        table.to_csv(out_path)
        print(f"Tabla guardada: {out_path}")

## 7. Estructura final de salidas

Las exportaciones quedan centralizadas en:

```
02a_params/
  02a_dasboard_outs/
    02a_dashboard/
      figures/
      tables/
```

In [ ]:
from datetime import datetime

figure_count = sum(1 for p in FIGURES_DIR.rglob("*") if p.is_file()) if FIGURES_DIR.exists() else 0
table_count = sum(1 for p in TABLES_DIR.rglob("*") if p.is_file()) if TABLES_DIR.exists() else 0

print(f"Figuras generadas: {figure_count}")
print(f"Tablas generadas: {table_count}")
print(f"Directorio base: {OUTPUT_BASE.resolve()}")

## 8. Conclusiones

- Las correlaciones top destacan métricas de eficiencia ofensiva y control del balón como principales impulsores de `W_PCT` y `PLUS_MINUS`.
- Los insights de puntos elevados con bajo porcentaje de victorias ayudan a aislar perfiles de volumen sin impacto colectivo.
- El ranking por `PLUS_MINUS` por partido permite identificar rápidamente rotaciones con mejor diferencial acumulado.
- Las rutas ordenadas facilitan incorporar estas salidas en pipelines automatizados del proyecto 02a.

**Próximos pasos**
- Extender el mismo formato a los cuadernos `02a_*` restantes.
- Analizar resultados segmentados por posición/rol para reducir sesgos.
- Vigilar futuros cambios de esquema en el parquet para ajustar validaciones y evitar leakage.

## 9. Guía de estilo aplicada

- Celdas de código cortas, cada una resuelve una única tarea.
- Se añadieron títulos Markdown (`##`, `###`) entre pasos para mejorar la narrativa.
- Máximo de tres figuras exportadas con tipografía tipo Computer Modern y sin dependencias de seaborn.
- Todas las rutas y nombres de archivo son relativos, en `snake_case` y con salidas centralizadas.

## 10. Compatibilidad y reproducibilidad

La siguiente celda registra versiones de librerías base (`pandas`, `numpy`, `matplotlib`) y la fecha de ejecución. Si el esquema del parquet cambia, las validaciones previas marcarán las secciones afectadas sin detener el cuaderno.

In [ ]:
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"Fecha de ejecución: {datetime.now().isoformat(timespec='seconds')}")